In [ ]:
import base64,hashlib,io,json,os,pathlib,runpy,sys,zipfile
for key in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):os.environ[key]='2'
target=pathlib.Path('/kaggle/working/exp042-payload')
assert not target.exists(),'Refusing repeated execution'
target.mkdir()
raw=base64.b64decode('...')
assert hashlib.sha256(raw).hexdigest()=='8df94080bf7af76eaa52a99285ad75287ff69ae1d0f6678f4b7f61d257222fb3'
with zipfile.ZipFile(io.BytesIO(raw)) as bundle:
 for name in bundle.namelist():
  assert pathlib.PurePosixPath(name).name==name and ':' not in name
  (target/name).write_bytes(bundle.read(name))
protocol=json.loads((target/'protocol.json').read_text())
for name in ('train.csv','extra.csv','folds.csv'):
 matches=[p for p in pathlib.Path('/kaggle/input').rglob(name) if hashlib.sha256(p.read_bytes()).hexdigest()==protocol['input_hashes'][name]]
 assert len(matches)==1,('Expected one frozen input',name,len(matches))
 (target/name).write_bytes(matches[0].read_bytes())
for name,digest in protocol['input_hashes'].items():assert hashlib.sha256((target/name).read_bytes()).hexdigest()==digest,name
sys.path.insert(0,str(target))
print('EXP042 frozen payload verified. CPU only, no network, no holdout text.',flush=True)
try:
 runpy.run_path(str(target/'guard.py'),run_name='__main__')
finally:
 root=pathlib.Path('/kaggle/working/results')
 if root.exists():
  with zipfile.ZipFile('/kaggle/working/exp042-results.zip','w',zipfile.ZIP_DEFLATED) as z:
   for p in sorted(root.rglob('*')):
    if p.is_file():z.write(p,p.relative_to(root).as_posix())
